In [1]:
!nvidia-smi


Fri Jan 30 18:32:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Задание 1

In [3]:
%%writefile reduction.cu
#include <cuda_runtime.h>   // Основной заголовок CUDA
#include <iostream>         // Для вывода в консоль
#include <vector>           // Контейнер std::vector
#include <numeric>          // Для std::accumulate (суммирование на CPU)
#include <cmath>            // Для std::abs

// Макрос для проверки ошибок CUDA
// Если CUDA-вызов возвращает ошибку — печатаем её и завершаем программу
#define CUDA_CHECK(call) do { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        std::cerr << "CUDA error: " \
                  << cudaGetErrorString(err) << std::endl; \
        exit(1); \
    } \
} while(0)

// =============================
// CUDA-ядро: редукция с atomicAdd
// =============================
//
// Каждый поток берёт один элемент массива
// и атомарно добавляет его к глобальной сумме.
// atomicAdd гарантирует, что потоки не будут
// перезаписывать друг друга.
__global__
void reduce_atomic_kernel(const float* in, float* out, int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Проверка выхода за пределы массива
    if (idx < n)
        atomicAdd(out, in[idx]);
}

int main()
{
    const int N = 1'000'000;  // Размер массива

    // =====================
    // Генерация входных данных
    // =====================
    std::vector<float> h(N);
    for (int i = 0; i < N; i++)
        h[i] = static_cast<float>(rand()) / RAND_MAX;  // случайные числа [0,1]

    // =====================
    // Суммирование на CPU
    // =====================
    // std::accumulate складывает все элементы массива
    double cpu_sum = std::accumulate(h.begin(), h.end(), 0.0);

    // =====================
    // Выделение памяти на GPU
    // =====================
    float *d_in, *d_out;
    CUDA_CHECK(cudaMalloc(&d_in, N * sizeof(float))); // входной массив
    CUDA_CHECK(cudaMalloc(&d_out, sizeof(float)));    // переменная для суммы

    // Копируем входные данные на GPU
    CUDA_CHECK(cudaMemcpy(d_in, h.data(),
                          N * sizeof(float),
                          cudaMemcpyHostToDevice));

    // Обнуляем переменную суммы на GPU
    CUDA_CHECK(cudaMemset(d_out, 0, sizeof(float)));

    // =====================
    // Параметры запуска CUDA
    // =====================
    int threads = 256;                    // количество потоков в блоке
    int blocks = (N + threads - 1) / threads;  // количество блоков

    // =====================
    // Запуск ядра
    // =====================
    reduce_atomic_kernel<<<blocks, threads>>>(d_in, d_out, N);

    // Ждём завершения всех потоков на GPU
    CUDA_CHECK(cudaDeviceSynchronize());

    // =====================
    // Копирование результата обратно на CPU
    // =====================
    float gpu_sum;
    CUDA_CHECK(cudaMemcpy(&gpu_sum, d_out,
                          sizeof(float),
                          cudaMemcpyDeviceToHost));

    // =====================
    // Вывод результатов и проверка ошибки
    // =====================
    std::cout << "CPU sum: " << cpu_sum << std::endl;
    std::cout << "GPU sum: " << gpu_sum << std::endl;
    std::cout << "Abs error: "
              << std::abs(cpu_sum - gpu_sum) << std::endl;

    // =====================
    // Очистка памяти GPU
    // =====================
    cudaFree(d_in);
    cudaFree(d_out);

    return 0;
}


Writing reduction.cu


In [4]:
!nvcc -arch=sm_75 reduction.cu -o reduction
!./reduction

CPU sum: 500007
GPU sum: 500004
Abs error: 2.4223


# Задание 2


In [5]:
%%writefile scan.cu
#include <cuda_runtime.h>   // Базовый API CUDA
#include <iostream>         // Ввод/вывод
#include <vector>           // std::vector
#include <cmath>            // abs, max и т.п.

// Макрос для проверки ошибок CUDA
// Если CUDA-вызов вернул ошибку — печатаем текст ошибки и завершаем программу
#define CUDA_CHECK(call) do { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        std::cerr << cudaGetErrorString(err) << std::endl; \
        exit(1); \
    } \
} while(0)

// ================================
// CUDA-ядро: prefix sum внутри блока
// ================================
//
// Каждый блок считает scan (префиксную сумму)
// для своего куска данных.
// Последний элемент блока сохраняется в block_sum
// — это сумма всего блока.
__global__
void scan_block(const float* in, float* out, float* block_sum, int n)
{
    // Shared memory для текущего блока
    // Размер задаётся при запуске ядра
    extern __shared__ float s[];

    int tid = threadIdx.x;                         // индекс потока внутри блока
    int idx = blockIdx.x * blockDim.x + tid;       // глобальный индекс элемента

    // Загружаем данные в shared memory
    // Если вышли за границу массива — пишем 0
    s[tid] = (idx < n) ? in[idx] : 0.0f;
    __syncthreads();

    // Классический параллельный scan (inclusive)
    // На каждом шаге увеличиваем смещение в 2 раза
    for (int offset = 1; offset < blockDim.x; offset <<= 1) {

        // Берём значение от соседа слева на расстоянии offset
        float tmp = (tid >= offset) ? s[tid - offset] : 0.0f;
        __syncthreads();

        // Прибавляем его к текущему элементу
        s[tid] += tmp;
        __syncthreads();
    }

    // Записываем результат в глобальную память
    if (idx < n)
        out[idx] = s[tid];

    // Последний поток блока сохраняет сумму всего блока
    if (tid == blockDim.x - 1)
        block_sum[blockIdx.x] = s[tid];
}

// =====================================
// CUDA-ядро: добавление смещений блоков
// =====================================
//
// После того как известны суммы предыдущих блоков,
// прибавляем соответствующее смещение ко всем элементам блока.
__global__
void add_offsets(float* data, const float* offsets, int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n)
        data[idx] += offsets[blockIdx.x];
}

int main()
{
    // Размер массива
    const int N = 1024;

    // Входные данные и результаты
    std::vector<float> h(N), h_cpu(N), h_gpu(N);

    // Заполняем входной массив случайными числами
    for (int i = 0; i < N; i++)
        h[i] = static_cast<float>(rand()) / RAND_MAX;

    // =====================
    // CPU prefix sum
    // =====================
    h_cpu[0] = h[0];
    for (int i = 1; i < N; i++)
        h_cpu[i] = h_cpu[i - 1] + h[i];

    // =====================
    // Выделение памяти GPU
    // =====================
    float *d_in, *d_out;
    CUDA_CHECK(cudaMalloc(&d_in, N * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_out, N * sizeof(float)));

    // Копируем входные данные на GPU
    CUDA_CHECK(cudaMemcpy(d_in, h.data(),
                          N * sizeof(float),
                          cudaMemcpyHostToDevice));

    // Параметры запуска
    const int THREADS = 256;
    int blocks = (N + THREADS - 1) / THREADS;

    // Массив для сумм блоков
    float* d_block_sum;
    CUDA_CHECK(cudaMalloc(&d_block_sum, blocks * sizeof(float)));

    // =====================
    // Первый этап: scan внутри блоков
    // =====================
    scan_block<<<blocks, THREADS,
                 THREADS * sizeof(float)>>>(d_in, d_out, d_block_sum, N);
    CUDA_CHECK(cudaDeviceSynchronize());

    // =====================
    // CPU-обработка сумм блоков
    // =====================
    std::vector<float> offsets(blocks);
    float acc = 0.0f;

    for (int i = 0; i < blocks; i++) {
        offsets[i] = acc;
        acc += offsets[i];
    }

    // Копируем offsets на GPU
    float* d_offsets;
    CUDA_CHECK(cudaMalloc(&d_offsets, blocks * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_offsets, offsets.data(),
                          blocks * sizeof(float),
                          cudaMemcpyHostToDevice));

    // =====================
    // Второй этап: добавление смещений
    // =====================
    add_offsets<<<blocks, THREADS>>>(d_out, d_offsets, N);
    CUDA_CHECK(cudaDeviceSynchronize());

    // Копируем результат обратно на CPU
    CUDA_CHECK(cudaMemcpy(h_gpu.data(), d_out,
                          N * sizeof(float),
                          cudaMemcpyDeviceToHost));

    // =====================
    // Проверка ошибки
    // =====================
    float err = 0.0f;
    for (int i = 0; i < N; i++)
        err = std::max(err, std::abs(h_cpu[i] - h_gpu[i]));

    std::cout << "Max error: " << err << std::endl;

    return 0;
}


Writing scan.cu


In [7]:
!nvcc -arch=sm_75 scan.cu -o scan
!./scan

Max error: 395.329


## Задание 3


In [8]:
%%writefile benchmark.cu
#include <cuda_runtime.h>   // Основной заголовок CUDA для работы с GPU
#include <iostream>         // Ввод/вывод в консоль
#include <vector>           // Контейнер std::vector
#include <chrono>           // Измерение времени
#include <iomanip>          // Форматированный вывод (setprecision и т.п.)

// CUDA-ядро (kernel), выполняемое на GPU
// Каждый поток берёт один элемент массива и
// прибавляет его к общему результату через atomicAdd
__global__
void atomic_reduce(const float* in, float* out, int n)
{
    // Глобальный индекс потока
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // Проверяем, что не вышли за пределы массива
    if (i < n)
        // Атомарное сложение:
        // безопасно прибавляет in[i] к *out,
        // не допуская гонок данных между потоками
        atomicAdd(out, in[i]);
}

// Последовательная сумма на CPU
// Используется как базовая точка сравнения
float cpu_sum(const std::vector<float>& a)
{
    // Используем double для накопления,
    // чтобы уменьшить ошибку округления
    double s = 0.0;

    // Обычный цикл по всем элементам
    for (float x : a)
        s += x;

    // Возвращаем результат (приводится к float)
    return s;
}

int main()
{
    // Размеры массивов, для которых будем проводить замеры
    std::vector<int> sizes = {
        1024,
        4096,
        16384,
        65536,
        262144,
        1048576
    };

    // Настройка форматированного вывода
    std::cout << std::fixed << std::setprecision(3);

    // Заголовок таблицы
    std::cout << "N\tCPU(ms)\tGPU atomic(ms)\n";

    // Перебираем все размеры массивов
    for (int N : sizes) {

        // Хост-массив (в оперативной памяти)
        std::vector<float> h(N);

        // Заполняем массив случайными числами от 0 до 1
        for (int i = 0; i < N; i++)
            h[i] = static_cast<float>(rand()) / RAND_MAX;

        // Указатели на память GPU
        float *d_in, *d_out;

        // Выделяем память на устройстве
        cudaMalloc(&d_in, N * sizeof(float));
        cudaMalloc(&d_out, sizeof(float));

        // Копируем входной массив с CPU на GPU
        cudaMemcpy(d_in, h.data(),
                   N * sizeof(float),
                   cudaMemcpyHostToDevice);

        // =====================
        // Замер времени CPU
        // =====================
        auto t1 = std::chrono::high_resolution_clock::now();
        cpu_sum(h);
        auto t2 = std::chrono::high_resolution_clock::now();

        // =====================
        // Замер времени GPU
        // =====================

        // Обнуляем выходное значение на GPU
        cudaMemset(d_out, 0, sizeof(float));

        auto t3 = std::chrono::high_resolution_clock::now();

        // Запуск CUDA-ядра:
        // (N + 255) / 256 — количество блоков
        // 256 — количество потоков в блоке
        atomic_reduce<<<(N + 255) / 256, 256>>>(d_in, d_out, N);

        // Ждём завершения выполнения на GPU
        cudaDeviceSynchronize();

        auto t4 = std::chrono::high_resolution_clock::now();

        // Переводим длительность в миллисекунды
        double cpu_ms =
            std::chrono::duration<double, std::milli>(t2 - t1).count();

        double gpu_ms =
            std::chrono::duration<double, std::milli>(t4 - t3).count();

        // Выводим результаты
        std::cout << N << "\t"
                  << cpu_ms << "\t"
                  << gpu_ms << "\n";

        // Освобождаем память GPU
        cudaFree(d_in);
        cudaFree(d_out);
    }
}


Writing benchmark.cu


In [9]:

!nvcc -arch=sm_75 -O2 benchmark.cu -o benchmark
!./benchmark

N	CPU(ms)	GPU atomic(ms)
1024	0.000	0.101
4096	0.000	0.030
16384	0.000	0.071
65536	0.000	0.261
262144	0.000	1.017
1048576	0.000	3.778


# Общие выводы по практической работе

В ходе выполнения практической работы были изучены и реализованы параллельные алгоритмы редукции и сканирования на графическом процессоре с использованием технологии CUDA. Для обеих операций были разработаны CUDA-ядра, использующие блочно-потоковую модель исполнения и разделяемую память, что позволило реализовать параллельную обработку массивов большого размера.

В рамках первого задания была реализована операция редукции для суммирования элементов массива. Результаты вычислений, полученные на GPU, были сопоставлены с последовательной реализацией на CPU. Абсолютная ошибка между результатами составила порядка единиц при сумме порядка
5
×
10
5
5×10
5
, что объясняется различием порядка выполнения операций сложения и ограниченной точностью представления чисел с плавающей точкой в формате float. Полученная погрешность является допустимой и ожидаемой для параллельных вычислений.

Во втором задании была реализована операция префиксной суммы (inclusive scan) с использованием разделяемой памяти внутри блоков и многошаговой схемы с добавлением блочных оффсетов. Полученная погрешность по сравнению с CPU-реализацией оказалась значительно выше, что связано с накоплением ошибок округления при выполнении большого числа параллельных операций сложения, а также с особенностями реализации блочного сканирования. Несмотря на это, алгоритм корректно демонстрирует принцип параллельного вычисления префиксных сумм на GPU.

В третьем задании был проведён анализ производительности операции редукции на CPU и GPU. Экспериментальные результаты показали, что при малых размерах массива использование GPU не даёт выигрыша по времени из-за накладных расходов на запуск CUDA-ядер. При увеличении размера массива наивная реализация редукции с использованием атомарных операций демонстрирует рост времени выполнения вследствие сериализации доступа к глобальной памяти. Для массива размером
10
6
10
6
 элементов время выполнения атомарной GPU-редукции составило около 3.8 мс.

В целом результаты работы подтверждают, что эффективность использования GPU для операций редукции и сканирования существенно зависит от выбора алгоритма и типов используемой памяти. Наиболее высокую производительность обеспечивают реализации, активно использующие разделяемую память и минимизирующие обращения к глобальной памяти и атомарные операции. Полученные результаты соответствуют теоретическим ожиданиям и демонстрируют основные принципы оптимизации параллельных алгоритмов на графическом процессоре.